In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set visualization style
sns.set_theme(style="whitegrid")

In [2]:
# Read the data from the Dataset folder
df = pd.read_csv('../Dataset/movie_metadata.csv')

# Drop rows where there is no imdb_score (our target)
df = df.dropna(subset=['imdb_score'])

# Create the Target Variable based on project rules
bins = [0, 3.0, 6.0, 10.0]
labels = ['Flop', 'Average', 'Hit']
df['success_category'] = pd.cut(df['imdb_score'], bins=bins, labels=labels, include_lowest=True)

# Drop the original imdb_score and useless identifiers
df = df.drop(columns=['imdb_score', 'movie_imdb_link'])

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0


In [4]:
# 1. Handle Missing Values
# Fill numeric NaNs with the median to avoid outliers skewing data
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical NaNs with "Unknown" or mode
categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# 2. Drop Highly Correlated/Redundant Columns (Addressing Multicollinearity)
# cast_total_facebook_likes is heavily correlated with actor_1_facebook_likes
df = df.drop(columns=['cast_total_facebook_likes'])

# 3. Encode Categorical Variables
le = LabelEncoder()
for col in categorical_cols:
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])

Categorical
color              19
director_name     104
actor_2_name       13
genres              0
actor_1_name        7
movie_title         0
actor_3_name       23
plot_keywords     153
language           14
country             5
content_rating    303
dtype: int64

Numerical
num_critic_for_reviews        50
duration                      15
director_facebook_likes      104
actor_3_facebook_likes        23
actor_1_facebook_likes         7
gross                        884
num_voted_users                0
cast_total_facebook_likes      0
facenumber_in_poster          13
num_user_for_reviews          21
budget                       492
title_year                   108
actor_2_facebook_likes        13
imdb_score                     0
aspect_ratio                 329
movie_facebook_likes           0
dtype: int64


In [ ]:
plt.figure(figsize=(15, 5))

# Plot 1: Target Distribution
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='success_category', palette='viridis', order=['Flop', 'Average', 'Hit'])
plt.title('Distribution of Movie Success')
plt.ylabel('Number of Movies')

# Plot 2: Budget vs. Success Category
plt.subplot(1, 2, 2)
# Using log scale for budget because movie budgets vary wildly
sns.boxplot(data=df, x='success_category', y='budget', palette='viridis', order=['Flop', 'Average', 'Hit'])
plt.yscale('log')
plt.title('Movie Budget by Success Category (Log Scale)')

plt.tight_layout()
plt.show()

In [ ]:
# Define Features (X) and Target (y)
X = df.drop(columns=['success_category'])
y = df['success_category']

# Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the numeric features so large numbers (like budget) don't overpower small numbers (like duration)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Initialize and train the Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

# Make Predictions
y_pred = rf_model.predict(X_test_scaled)

# 1. Print Classification Report
print("--- Random Forest Classification Report ---")
print(classification_report(y_test, y_pred))

# 2. Plot Confusion Matrix
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, y_pred, labels=['Flop', 'Average', 'Hit'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Flop', 'Average', 'Hit'], yticklabels=['Flop', 'Average', 'Hit'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# 3. Plot Feature Importances (What makes a movie successful?)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-10:] # Top 10 features

plt.figure(figsize=(8, 5))
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.title('Top 10 Most Important Features for Predicting Movie Success')
plt.show()

In [ ]:
import joblib
import os

# 1. Create the Models directory if it doesn't exist yet
os.makedirs('../Models', exist_ok=True)

# 2. Export the trained Random Forest model to that folder
model_path = '../Models/rf_movie_model.pkl'
joblib.dump(rf_model, model_path)

print(f"✅ Success! Your model has been saved to: {model_path}")